# Module 09: MCP for AI Agents

## What You'll Learn

- What MCP (Model Context Protocol) is
- Feast as an MCP server: governed context layer for AI agents
- How `fastapi_mcp` automatically exposes Feast endpoints as MCP tools
- Agent discovery of features without hardcoded definitions
- Configuration: `pip install feast[mcp]`
- HTTP support added in v0.62.0

---

> **🗺️ DATA STRATEGY**: **Pillar 5 — Unified Data & AI Experience**. MCP enables **Context Engineering** (Data Scenario #11). The data strategy envisions MCP-based agent discovery as **Phase 3** of the data abstraction layer — agents dynamically discover governed features rather than hardcoding schemas.

## What Is MCP?

**Model Context Protocol (MCP)** is an open standard for connecting AI agents to external data sources and tools. Think of it as a USB-C port for AI — one protocol, many integrations.

```
MCP Architecture:

  AI Agent (Claude, custom LLM agent)
    ↔ MCP Client (in the agent runtime)
    ↔ MCP Server (exposes tools & resources)
    ↔ Backend System (Feast feature store, databases, APIs)
```

MCP defines three primitives:

| Primitive | Purpose | Feast Example |
|-----------|---------|---------------|
| **Tools** | Actions the agent can invoke | `get_online_features`, `list_feature_views` |
| **Resources** | Data the agent can read | Feature view schemas, entity definitions |
| **Prompts** | Templated interactions | "Describe available features for customer scoring" |

> **🔮 UPSTREAM**: Feast MCP server is available upstream only. HTTP transport support was added in v0.62.0. **Not downstream in RHOAI yet.**
>
> **⚠️ GAP**: Feast MCP server exists upstream but is **not available in RHOAI**. Agentic workflows that discover features dynamically cannot be deployed on-platform today.

## Feast as MCP Server

Feast exposes its feature server endpoints as MCP tools via **`fastapi_mcp`**. When MCP is enabled, the Feast feature server automatically registers:

- **Feature discovery tools** — list feature views, entities, data sources
- **Feature retrieval tools** — get online features, get historical features
- **Schema resources** — feature view definitions, field types, descriptions

This means an AI agent can:
1. **Discover** what features exist (no hardcoded feature lists)
2. **Understand** feature schemas and semantics
3. **Retrieve** features at inference time with governed access (RBAC from Module 10)

```
Agent Workflow:
  1. Agent connects to Feast MCP server
  2. Agent calls list_feature_views() → discovers credit_scoring features
  3. Agent calls get_online_features(entity_keys=[123]) → retrieves live data
  4. Agent uses features in reasoning / model inference
```

> **🗺️ DATA STRATEGY**: This is the "Context Engineering" pattern from Scenario #11. Instead of copying feature definitions into agent prompts, agents query the governed feature store at runtime. Phase 3 of the data abstraction layer roadmap.

## Installation & Configuration

Install Feast with MCP extras:

```bash
pip install "feast[mcp]"
```

Enable MCP in `feature_store.yaml`:

```yaml
project: mcp_demo
provider: local

# ... registry, stores ...

feature_server:
  enabled: true
  port: 6566
  mcp:
    enabled: true
    transport: http          # HTTP transport (v0.62.0+)
    # transport: stdio       # Alternative: stdio for local agent integration
    endpoint: /mcp           # MCP endpoint path
```

Start the feature server with MCP:

```bash
feast serve --mcp
# Feature server starts on :6566
# MCP endpoint available at http://localhost:6566/mcp
```

> **📍 RHOAI STATUS**: MCP server is **not downstream in RHOAI**. This configuration works in local/upstream environments only.
>
> **🖥️ UI**: No MCP-specific UI in Feast. Use MCP client tools (Claude Desktop, custom agent frameworks) to interact with the MCP endpoint.

In [ ]:
# Install Feast with MCP support
# Run in terminal or uncomment below:

# !pip install "feast[mcp]"

import subprocess
import sys

def check_mcp_installed():
    try:
        import fastapi_mcp
        print(f"fastapi_mcp available: {fastapi_mcp.__version__ if hasattr(fastapi_mcp, '__version__') else 'installed'}")
        return True
    except ImportError:
        print("fastapi_mcp not installed. Run: pip install 'feast[mcp]'")
        return False

check_mcp_installed()

In [ ]:
# Create feature_store.yaml with MCP enabled
import os
import yaml

os.makedirs("feature_repo", exist_ok=True)
os.makedirs("data", exist_ok=True)

config = {
    "project": "mcp_demo",
    "provider": "local",
    "registry": {
        "registry_type": "sql",
        "path": "sqlite:///data/registry.db",
    },
    "offline_store": {"type": "duckdb"},
    "online_store": {
        "type": "sqlite",
        "path": "data/online.db",
    },
    "entity_key_serialization_version": 3,
    "feature_server": {
        "enabled": True,
        "port": 6566,
        "mcp": {
            "enabled": True,
            "transport": "http",
            "endpoint": "/mcp",
        },
    },
}

with open("feature_repo/feature_store.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("feature_store.yaml created with MCP enabled")
print("Start server: feast serve --mcp")
print("MCP endpoint: http://localhost:6566/mcp")

In [ ]:
# Start the Feast feature server with MCP (run in terminal)
# feast serve --mcp

print("To start the MCP-enabled feature server:")
print("")
print("  cd feature_repo")
print("  feast apply                    # Register feature definitions")
print("  feast serve --mcp              # Start server with MCP on :6566")
print("")
print("The server exposes these MCP tools automatically:")
print("  - list_feature_views")
print("  - list_entities")
print("  - get_online_features")
print("  - get_historical_features")
print("  - push")
print("")
print("Agents discover tools via MCP protocol — no hardcoded API calls needed.")

In [ ]:
# Example: How an AI agent discovers and uses Feast features via MCP
# This simulates the agent interaction pattern (pseudocode for MCP client)

agent_workflow = '''
# === Agent connects to Feast MCP server ===
mcp_client = MCPClient("http://localhost:6566/mcp")

# Step 1: Discover available tools (no hardcoded feature lists)
tools = mcp_client.list_tools()
print(f"Available tools: {[t.name for t in tools]}")
# Output: ['list_feature_views', 'get_online_features', 'list_entities', ...]

# Step 2: Discover feature views
feature_views = mcp_client.call_tool("list_feature_views", {})
print(f"Feature views: {feature_views}")
# Output: ['credit_history', 'transaction_aggregates', 'customer_profile']

# Step 3: Agent decides which features it needs (autonomous reasoning)
agent_reasoning = """
The user asked about customer credit risk.
I need credit_history and transaction_aggregates features.
Entity key is customer_id=42.
"""

# Step 4: Retrieve features via MCP tool
features = mcp_client.call_tool("get_online_features", {
    "feature_refs": [
        "credit_history:credit_score",
        "credit_history:delinquency_count",
        "transaction_aggregates:avg_amount_30d",
    ],
    "entity_rows": [{"customer_id": 42}],
})
print(f"Retrieved features: {features}")

# Step 5: Agent uses features in its response
# "Customer 42 has a credit score of 720 with 0 delinquencies..."
'''

print(agent_workflow)

In [ ]:
# MCP HTTP client example (v0.62.0+ HTTP transport)
# Demonstrates programmatic MCP interaction without stdio

import json

mcp_request_example = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "get_online_features",
        "arguments": {
            "feature_refs": ["credit_history:credit_score"],
            "entity_rows": [{"customer_id": 42}],
        },
    },
}

print("HTTP MCP request to Feast feature server:")
print(f"POST http://localhost:6566/mcp")
print(json.dumps(mcp_request_example, indent=2))
print("")
print("# With requests library:")
print("# response = requests.post('http://localhost:6566/mcp', json=mcp_request_example)")
print("# features = response.json()['result']")

## MCP + RBAC: Governed Agent Access

When combined with Feast RBAC (Module 10), MCP provides **governed context** for agents:

```
Agent Request → MCP Server → RBAC Check → Feature Retrieval
                         ↓
                   Permission denied?
                   Agent gets 403, not raw data
```

This is critical for enterprise deployments where agents must not access features outside their authorization scope.

> **🗺️ DATA STRATEGY**: MCP + RBAC = governed context engineering. Agents discover features dynamically but only access what their identity permits. This is the Phase 3 vision for the unified data & AI experience.

## Key Takeaways

1. **MCP is the standard** for connecting AI agents to external data sources
2. **Feast exposes MCP tools automatically** via `fastapi_mcp` when `--mcp` is enabled
3. **Agents discover features dynamically** — no hardcoded feature definitions in prompts
4. **HTTP transport** (v0.62.0+) enables remote agent connections
5. **Not downstream in RHOAI** — upstream only; on-platform agentic workflows cannot use this yet
6. **Phase 3 of data abstraction** — Context Engineering via MCP-based discovery

## What's Next

- **Module 10**: RBAC & Governance — securing MCP agent access
- **Module 08**: RAG & Vector Search — another context source for agents
- **Module 15**: Knowledge Retrieval — OGX as the primary agentic RAG runtime on RHOAI